In [1]:
import pandas as pd
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import textstat

# Download required NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')

# Function to compute scores
def compute_scores(ground_truth, summary):
    # Tokenize the inputs
    ground_truth_tokens = ground_truth.split()
    summary_tokens = summary.split()

    # Compute BLEU score
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([ground_truth_tokens], summary_tokens, smoothing_function=smoothing_function)
    
    # Compute METEOR score
    meteor = meteor_score([ground_truth_tokens], summary_tokens)
    
    # Compute ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ground_truth, summary)
    rouge1 = scores['rouge1'].fmeasure
    rouge2 = scores['rouge2'].fmeasure
    rougeL = scores['rougeL'].fmeasure
    
    # Compute Flesch Reading Ease score
    flesch_reading_ease = textstat.flesch_reading_ease(summary)
    
    return bleu_score, meteor, rouge1, rouge2, rougeL, flesch_reading_ease

def main(input_csv, ground_truth_text, output_csv):
    # Read input CSV
    df = pd.read_csv(input_csv)
    
    # Initialize lists to store scores for BART summaries
    bart_bleu_scores = []
    bart_meteor_scores = []
    bart_rouge1_scores = []
    bart_rouge2_scores = []
    bart_rougeL_scores = []
    bart_flesch_reading_ease_scores = []

    # Initialize lists to store scores for BERT summaries
    bert_bleu_scores = []
    bert_meteor_scores = []
    bert_rouge1_scores = []
    bert_rouge2_scores = []
    bert_rougeL_scores = []
    bert_flesch_reading_ease_scores = []
    
    # Compute scores for BART summaries
    for summary in df['Summary (BART) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bart_bleu_scores.append(bleu)
        bart_meteor_scores.append(meteor)
        bart_rouge1_scores.append(rouge1)
        bart_rouge2_scores.append(rouge2)
        bart_rougeL_scores.append(rougeL)
        bart_flesch_reading_ease_scores.append(flesch_reading_ease)

    # Compute scores for BERT summaries
    for summary in df['Summary (BERT) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bert_bleu_scores.append(bleu)
        bert_meteor_scores.append(meteor)
        bert_rouge1_scores.append(rouge1)
        bert_rouge2_scores.append(rouge2)
        bert_rougeL_scores.append(rougeL)
        bert_flesch_reading_ease_scores.append(flesch_reading_ease)
    
    # Add scores to DataFrame
    df['BLEU (BART)'] = bart_bleu_scores
    df['METEOR (BART)'] = bart_meteor_scores
    df['ROUGE-1 (BART)'] = bart_rouge1_scores
    df['ROUGE-2 (BART)'] = bart_rouge2_scores
    df['ROUGE-L (BART)'] = bart_rougeL_scores
    df['Flesch Reading Ease (BART)'] = bart_flesch_reading_ease_scores

    df['BLEU (BERT)'] = bert_bleu_scores
    df['METEOR (BERT)'] = bert_meteor_scores
    df['ROUGE-1 (BERT)'] = bert_rouge1_scores
    df['ROUGE-2 (BERT)'] = bert_rouge2_scores
    df['ROUGE-L (BERT)'] = bert_rougeL_scores
    df['Flesch Reading Ease (BERT)'] = bert_flesch_reading_ease_scores
    
    # Save DataFrame to new CSV
    df.to_csv(output_csv, index=False)
    
if __name__ == "__main__":
    input_csv = 'CSV/Refining/UnicodeR1JANUS.csv'  
    ground_truth_text = """
The agent-based model in this study investigates megaherbivore extinction influenced by human hunting pressure and environmental changes. Simulating an environment with grassland patches, ungulates, and human hunters, it aims to understand how hunting behaviors, herd demography, and environmental conditions interact to influence extinction events . Key parameters include 20 initial foragers, processing costs of 3 out of 20 for females and 6 out of 20 for males, and a maximum animal lifespan of 10,220 ticks. The initial prey count is 250, with a food value of 30 out of 100 for males and 27 out of 100 for females. Every three ticks, one third of the grass grows back. Foragers, designed to be immortal, have a foraging radius of 6 patches and an energy cost of 1 unit per tick. Mortality rates are 0.025 out of 0.2 internally and 0.0025 out of 0.2 externally. The model runs for up to 73,000 ticks.

The number of foragers remains constant through a simulation run at 20 due to immortality. However, their total energy declines markedly, starting near zero and approaching -40,000 by the end, indicating a consistent energy deficit. Predation patterns show increasing kills of males, females, and postpartum females, peaking at approximately 700 ticks and plateauing between 100-120 kills per category. Birth counts among preys vary significantly, with some simulations showing frequent births and exhibiting others long intervals. The maximum recorded births in any simulation are around 10, while some show none.

Total energy for male and female animals declines from around 12,000 rapidly within  400 ticks, with a brief increase around 500 ticks before continuing to zero. The total energy of all animals follows this pattern, starting at 25,000 and declining steadily with slight increases before approaching zero. The percentages of green patches (i.e., with grass) fluctuates across simulation runs, starting at 32%, dropping to 23-27% at regular intervals, then stabilizing around 33% after the fourth decline, indicating a cyclical but resilient vegetation regrowth pattern.

"""
    output_csv = 'CSV/Refining/ScoresR1JANUS.csv'
    main(input_csv, ground_truth_text, output_csv)


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
